In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import RobustScaler, OneHotEncoder, FunctionTransformer, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from tensorflow.keras import layers, models, callbacks

In [3]:
df = pd.read_csv("data/unsw_nb_cleaned.csv")

X = df.drop(columns=["attack_cat"])
y = df["attack_cat"]

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 175341 entries, 0 to 175340
Data columns (total 16 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   proto       175341 non-null  object 
 1   state       175341 non-null  object 
 2   dur         175341 non-null  float64
 3   spkts       175341 non-null  int64  
 4   dpkts       175341 non-null  int64  
 5   sbytes      175341 non-null  int64  
 6   dbytes      175341 non-null  int64  
 7   sttl        175341 non-null  int64  
 8   dttl        175341 non-null  int64  
 9   sload       175341 non-null  float64
 10  dload       175341 non-null  float64
 11  sloss       175341 non-null  int64  
 12  dloss       175341 non-null  int64  
 13  ct_srv_src  175341 non-null  int64  
 14  ct_dst_ltm  175341 non-null  int64  
 15  attack_cat  175341 non-null  object 
dtypes: float64(3), int64(10), object(3)
memory usage: 21.4+ MB


In [5]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

num_classes = len(np.unique(y))

In [6]:
split = int(len(df) * 0.85)

X_train_raw = X.iloc[:split]
X_val_raw   = X.iloc[split:]

y_train_raw = y[:split]
y_val_raw   = y[split:]

In [7]:
cat_cols = ["proto", "state"]

num_cols = [
    "dur","spkts","dpkts","sbytes","dbytes",
    "sttl","dttl","sload","dload",
    "sloss","dloss","ct_srv_src","ct_dst_ltm"
]

skewed_cols = ["spkts","dpkts","sbytes","dbytes","sload","dload","sloss","dloss"]
normal_cols = list(set(num_cols) - set(skewed_cols))

log_tf = FunctionTransformer(np.log1p)

preprocessor = ColumnTransformer([
    ("skewed", Pipeline([
        ("log", log_tf),
        ("scale", RobustScaler())
    ]), skewed_cols),

    ("normal", RobustScaler(), normal_cols),

    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
])

preprocessor.fit(X_train_raw)

X_train = preprocessor.transform(X_train_raw).astype(np.float32)
X_val   = preprocessor.transform(X_val_raw).astype(np.float32)

In [22]:
os.makedirs("../artifacts/odd", exist_ok=True)

with open("../artifacts/odd/preprocessor.pkl", "wb") as f:
    pickle.dump(preprocessor, f)

with open("../artifacts/odd/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

feature_order = X.columns.tolist()
with open("../artifacts/odd/feature_order.pkl", "wb") as f:
    pickle.dump(feature_order, f)

In [8]:
T = 30
batch_size = 64

def window_generator(X, y, T, batch_size=64):
    n = len(X)

    while True:
        for i in range(0, n - T + 1, batch_size):
            X_batch, y_batch = [], []

            for j in range(i, min(i + batch_size, n - T + 1)):
                X_batch.append(X[j:j+T])
                #y_batch.append(y[j:j+T])
                y_batch.append(y[j+T-1])

            yield np.array(X_batch, dtype=np.float32), np.array(y_batch, dtype=np.int32)

In [9]:
train_gen = window_generator(X_train, y_train_raw, T, batch_size)
val_gen   = window_generator(X_val, y_val_raw, T, batch_size)

train_steps = (len(X_train) - T + 1) // batch_size
val_steps   = (len(X_val) - T + 1) // batch_size

In [10]:
def focal_loss(gamma=2.0):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_true_one_hot = tf.one_hot(y_true, depth=num_classes)

        pt = tf.reduce_sum(y_true_one_hot * y_pred, axis=-1)
        loss = -tf.pow(1. - pt, gamma) * tf.math.log(pt + 1e-9)

        return tf.reduce_mean(loss)
    return loss


"""
better 

def focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        y_true = tf.one_hot(tf.cast(y_true, tf.int32), depth=num_classes)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)

        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = alpha * tf.pow(1 - y_pred, gamma)

        return tf.reduce_mean(weight * cross_entropy)
    return loss

"""

'\nbetter \n\ndef focal_loss(gamma=2.0, alpha=0.25):\n    def loss(y_true, y_pred):\n        y_true = tf.one_hot(tf.cast(y_true, tf.int32), depth=num_classes)\n        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)\n\n        cross_entropy = -y_true * tf.math.log(y_pred)\n        weight = alpha * tf.pow(1 - y_pred, gamma)\n\n        return tf.reduce_mean(weight * cross_entropy)\n    return loss\n\n'

In [11]:
inp = layers.Input(shape=(T, X_train.shape[1]))

x = layers.Conv1D(64, 3, padding="same", activation="relu")(inp)
x = layers.BatchNormalization()(x)

x = layers.Bidirectional(
    layers.LSTM(64, return_sequences=True)
)(x)

attn = layers.MultiHeadAttention(num_heads=2, key_dim=32)(x, x)
x = layers.Add()([x, attn])
x = layers.LayerNormalization()(x)

x = layers.TimeDistributed(layers.Dense(64, activation="relu"))(x)

# ✅ TAKE ONLY LAST TIMESTEP
x = x[:, -1, :]

# ✅ SINGLE OUTPUT
out = layers.Dense(num_classes, activation="softmax")(x)

model = models.Model(inp, out)

In [12]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-4, clipnorm=1.0),
    loss=focal_loss(),
    metrics=["accuracy"]
)

In [13]:
history = model.fit(
    train_gen,
    steps_per_epoch=train_steps,
    validation_data=val_gen,
    validation_steps=val_steps,
    epochs=25,
    callbacks=[
        callbacks.EarlyStopping(patience=6, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(patience=3)
    ]
)

Epoch 1/25
2328/2328 ━━━━━━━━━━━━━━━━━━━━ 97s 40ms/step - accuracy: 0.8476 - loss: 0.2821 - val_accuracy: 0.9012 - val_loss: 0.1685 - learning_rate: 3.0000e-04
Epoch 2/25
2328/2328 ━━━━━━━━━━━━━━━━━━━━ 85s 37ms/step - accuracy: 0.8514 - loss: 0.3067 - val_accuracy: 0.9027 - val_loss: 0.1560 - learning_rate: 3.0000e-04
Epoch 3/25
2328/2328 ━━━━━━━━━━━━━━━━━━━━ 81s 35ms/step - accuracy: 0.8668 - loss: 0.2264 - val_accuracy: 0.9045 - val_loss: 0.1495 - learning_rate: 3.0000e-04
Epoch 4/25
2328/2328 ━━━━━━━━━━━━━━━━━━━━ 79s 34ms/step - accuracy: 0.8744 - loss: 0.2002 - val_accuracy: 0.9089 - val_loss: 0.1419 - learning_rate: 3.0000e-04
Epoch 5/25
2328/2328 ━━━━━━━━━━━━━━━━━━━━ 84s 36ms/step - accuracy: 0.8807 - loss: 0.1818 - val_accuracy: 0.9077 - val_loss: 0.1434 - learning_rate: 3.0000e-04
Epoch 6/25
2328/2328 ━━━━━━━━━━━━━━━━━━━━ 96s 41ms/step - accuracy: 0.8853 - loss: 0.1713 - val_accuracy: 0.9049 - val_loss: 0.1455 - learning_rate: 3.0000e-04
Epoch 7/25
2328/2328 ━━━━━━━━━━━━━━━━━━━

In [14]:
from sklearn.metrics import classification_report

y_pred = model.predict(val_gen, steps=val_steps)
y_pred = np.argmax(y_pred, axis=-1)

# flatten
y_true = []
for i in range(val_steps):
    _, batch_y = next(val_gen)
    y_true.append(batch_y)

y_true = np.concatenate(y_true).reshape(-1)
y_pred = y_pred.reshape(-1)

print(classification_report(y_true, y_pred))

410/410 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step
              precision    recall  f1-score   support

           0       0.47      0.28      0.35       125
           1       0.55      0.06      0.10       104
           2       0.37      0.45      0.41       805
           3       0.70      0.74      0.72      2451
           4       0.78      0.77      0.78      1534
           5       0.99      0.98      0.99     20248
           6       0.00      0.00      0.00         0
           7       0.70      0.75      0.73       840
           8       0.35      0.19      0.25        93
           9       0.00      0.00      0.00         9

    accuracy                           0.91     26209
   macro avg       0.49      0.42      0.43     26209
weighted avg       0.92      0.91      0.91     26209



d:\AegisGAN_baseClassifier\AegisENV\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\AegisGAN_baseClassifier\AegisENV\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\AegisGAN_baseClassifier\AegisENV\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\AegisGAN_bas

In [15]:
#step 16
model.save(r"Artifacts/final_model96.keras")

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import numpy as np
import pandas as pd
import pickle
import tensorflow as tf

app = FastAPI()

# -----------------------------
# LOAD ARTIFACTS
# -----------------------------
model = tf.keras.models.load_model("../artifacts/model.keras")

with open("../artifacts/preprocessor.pkl", "rb") as f:
    preprocessor = pickle.load(f)

with open("../artifacts/label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

with open("../artifacts/feature_order.pkl", "rb") as f:
    feature_order = pickle.load(f)

T = 30  # window size


# -----------------------------
# INPUT SCHEMA
# -----------------------------
class LogRequest(BaseModel):
    logs: list  # list of dicts


# -----------------------------
# WINDOW CREATION
# -----------------------------
def create_windows(X, T=30):
    windows = []
    indices = []

    n = len(X)

    for i in range(n):
        start = max(0, i - T + 1)
        window = X[start:i+1]

        # pad if needed
        if len(window) < T:
            pad = np.zeros((T - len(window), X.shape[1]))
            window = np.vstack([pad, window])

        windows.append(window)
        indices.append(i)

    return np.array(windows), indices


# -----------------------------
# INFERENCE ROUTE
# -----------------------------
@app.post("/predict")
def predict(request: LogRequest):

    # -------------------------
    # 1. Convert to DataFrame
    # -------------------------
    df = pd.DataFrame(request.logs)

    # enforce feature order
    df = df[feature_order]

    # -------------------------
    # 2. Preprocess
    # -------------------------
    X = preprocessor.transform(df).astype(np.float32)

    # -------------------------
    # 3. Create windows
    # -------------------------
    X_seq, idx_map = create_windows(X, T)

    # -------------------------
    # 4. Model prediction
    # -------------------------
    preds = model.predict(X_seq)  # (N, T, num_classes)

    # take last timestep prediction (log-level output)
    preds = preds[:, -1, :]

    # -------------------------
    # 5. Convert to labels + confidence
    # -------------------------
    pred_classes = np.argmax(preds, axis=1)
    confidence = np.max(preds, axis=1)

    labels = le.inverse_transform(pred_classes)

    # -------------------------
    # 6. Build response
    # -------------------------
    results = []

    for i, idx in enumerate(idx_map):
        results.append({
            "log_index": int(idx),
            "predicted_label": labels[i],
            "confidence": float(confidence[i])
        })

    return {"results": results}